In [1]:
!pip install keras-bert

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for keras-bert: filename=keras_bert-0.89.0-py3-none-any.whl size=33500 sha256=e0fe1f6b62d68b2f2f662fc010799fd468b9473327e53e61ff784c64626cf35e
  Stored in directory: /root/.cache/pip/wheels/89/dd/7a/e3c94de9a5b91bf45958aad780cadae9212a60c0dacd4576a4
  Created wheel for keras-transformer: filename=keras_transformer-0.40.0-py3-none-any.whl size=12285 sha256=493a3b2066cb124381a0ccf53d89704fae0570f42a55cad7e5e6a94a97002162
  Stored in directory: /root/.cache/pip/wheels/20/f6/8d/9063ab182cbf213b859ee9dfd602df75928007bbb916e8a2c7
  Created wheel for keras-embed-sim: filename=keras_embed_sim-0.10.0-py3-none-any.whl size=3944 sha256=a30657416b219916dbc4d16

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'
import argparse
from utils import (seq2tokens, get_token_dict, process_bert_tokens_batch, load_bert,
                          parse_fasta, annotate_predictions, best_predictions, seq_frames)
import tensorflow as tf
from logging import  info, getLogger, INFO, WARNING
import numpy as np
from random import sample
import json
import pkg_resources

<ipython-input-1-2bc217613f88>:13: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
MAX_SIZE = 1500
FIELD_LEN_THR = 50 # prevent overlong IDs
CLASS_LABELS = ["Cacao", "NotCacao"]
# Ruta al archivo FASTA
fasta_path = "/content/20cacao.fasta"

# Ruta al modelo BERT (.keras)
model_path = "/content/fine-tune-BERTaxCacao.keras"

# Opciones principales
verbose = False
output_file = None  # o por ejemplo: "resultados.tsv"
conf_matrix_file = "confidencias.json"  # o por ejemplo: "confidencias.json"

# Opciones de predicción
sequence_split = 'equal_chunks'  # opciones: 'equal_chunks', 'window'
chunk_predictions = False
running_window = False
running_window_stride = 90
custom_window_size = None  # por ejemplo: 1000
maximum_sequence_chunks = 151
output_ranks = ["Cacao", "NotCacao"]
no_confidence = False
batch_size = 16

In [3]:
def check_max_len_arg(value):
    value = int(value)
    if (value < 1 or value > MAX_SIZE):
        raise argparse.ArgumentTypeError(f'value has to be between 1 and {MAX_SIZE}')
    return value

In [4]:
def check_ranks(value):
    if (value not in CLASS_LABELS):
        raise argparse.ArgumentTypeError(f'output ranks have to be a combination of {set(CLASS_LABELS)}, '
                                         'other ranks are not predicted')
    return value

In [7]:
!unzip fine-tune-BERTaxCacao.keras model.weights.h5
!unzip fine-tune-BERTaxCacao.keras config.json

Archive:  fine-tune-BERTaxCacao.keras
 extracting: model.weights.h5        
Archive:  fine-tune-BERTaxCacao.keras
 extracting: config.json             


In [5]:
weights_path = 'model.weights.h5'
config_path = 'config.json'

In [6]:
import gc
import tensorflow as tf
getLogger().setLevel(INFO)
# 1. Cargar modelo y preparar datos
model = load_bert(config_path=config_path, weights_path=weights_path, compile_=True)
max_seq_len = 151 if custom_window_size is None else custom_window_size
token_dict = get_token_dict()
records = parse_fasta(fasta_path)
sequences = [r.seq for r in records]
record_ids = [r.id for r in records]
output_dir = "result_chunks"
os.makedirs(output_dir, exist_ok=True)
out = []

chunk_size = 200000
total = len(sequences)
preds_all = []

for i in range(0, total, chunk_size):
    print(f"\nProcesando secuencias {i} a {i+chunk_size}")

    seq_chunk = sequences[i:i+chunk_size]
    id_chunk = record_ids[i:i+chunk_size]

    tokenized_inputs = [seq2tokens(seq, token_dict, 500, max_length=max_seq_len) for seq in seq_chunk]
    x = process_bert_tokens_batch(tokenized_inputs)

    # Dataset y predict con batch dinámico
    batch_size = 1024
    min_batch_size = 16
    preds = None

    while batch_size >= min_batch_size:
        try:
            with tf.device("/GPU:0"):
                preds = model.predict(x, verbose=int(verbose), batch_size=batch_size)
            print(f"Chunk {i} OK con batch_size={batch_size}")
            break
        except tf.errors.ResourceExhaustedError:
            print(f"OOM con batch_size={batch_size}, probando con {batch_size // 2}")
            batch_size //= 2

    if preds is None:
        raise RuntimeError(f"No se pudo predecir el chunk {i} ni con batch_size={min_batch_size}")

    #preds_all.append(preds)
    np.save(f"{output_dir}/preds_chunk_{i}.npy", preds[1])
    np.save(f"{output_dir}/ids_chunk_{i}.npy", id_chunk)

    # Liberar memoria
    del x, tokenized_inputs, preds, id_chunk, seq_chunk
    tf.keras.backend.clear_session()
    gc.collect()

Ana test :: inputs: 
Tensor("Placeholder:0", shape=(None, 151, 256), dtype=float32)


/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:629: UserWarning: A total of 4 objects could not be loaded. Example error message for object <LayerNormalization name=MLM-Norm, built=True>:

Layer 'MLM-Norm' expected 2 variables, but received 0 variables during loading. Expected: ['gamma', 'beta']

List of objects that could not be loaded:
[<LayerNormalization name=MLM-Norm, built=True>, <EmbeddingSimilarity name=MLM-Sim, built=True>, <Dense name=NSP-Dense, built=True>, <Dense name=NSP, built=True>]
  warnings.warn(msg)
INFO:root:read in 2195521 sequences



Procesando secuencias 0 a 200000
Ana test :: inputs: 
Tensor("functional_1/MLM-Norm_1/add_1:0", shape=(1024, 151, 256), dtype=float32)
Ana test :: inputs: 
Tensor("functional_1/MLM-Norm_1/add_1:0", shape=(None, 151, 256), dtype=float32)
Chunk 0 OK con batch_size=1024

Procesando secuencias 200000 a 400000
Chunk 200000 OK con batch_size=1024

Procesando secuencias 400000 a 600000
Chunk 400000 OK con batch_size=1024

Procesando secuencias 600000 a 800000
Chunk 600000 OK con batch_size=1024

Procesando secuencias 800000 a 1000000
Chunk 800000 OK con batch_size=1024

Procesando secuencias 1000000 a 1200000
Chunk 1000000 OK con batch_size=1024

Procesando secuencias 1200000 a 1400000
Chunk 1200000 OK con batch_size=1024

Procesando secuencias 1400000 a 1600000
Chunk 1400000 OK con batch_size=1024

Procesando secuencias 1600000 a 1800000
Chunk 1600000 OK con batch_size=1024

Procesando secuencias 1800000 a 2000000
Chunk 1800000 OK con batch_size=1024

Procesando secuencias 2000000 a 2200000

In [7]:
import glob

all_preds = []
for file in sorted(glob.glob(f"{output_dir}/preds_chunk_*.npy")):
    all_preds.append(np.load(file))
final_preds = np.concatenate(all_preds, axis=0)
print("Total de predicciones:", len(final_preds))

Total de predicciones: 2195521


In [8]:
# ====================== #
# 🔽 OUTPUT DE LOS DATOS #
# ====================== #

import json

# Mapear índice a clase
labels = ["No Cacao", "Cacao"]
binary_preds = final_preds
# Obtener predicción más probable y su probabilidad
results = []

for i, prediction in enumerate(binary_preds):
    class_idx = np.argmax(prediction)  # 0 o 1
    confidence = prediction[class_idx]
    results.append((record_ids[i], labels[class_idx], confidence))

# Mostrar en consola si no hay output_file
if output_file is None:
    header = ["id", "predicted_class", "confidence"]
    max_lens = [max(len(str(r[i])) for r in results + [header]) for i in range(3)]
    row_format = ''.join([f'{{:<{w+2}}}' for w in max_lens])
    print(row_format.format(*header))
    for row in results:
        print(row_format.format(row[0], row[1], f"{row[2]:.2%}"))

# Guardar en archivo si se especifica
else:
    with open(output_file, 'w') as handle:
        handle.write("id\tpredicted_class\tconfidence\n")
        for rid, cls, conf in results:
            handle.write(f"{rid}\t{cls}\t{conf:.4f}\n")

# Guardar matriz completa (opcional)
if conf_matrix_file is not None:
    pred_json = {
        rid: {
            "Cacao": float(pred[1]),
            "NotCacao": float(pred[0]),
            "Predicted": labels[np.argmax(pred)]
        }
        for rid, pred in zip(record_ids, binary_preds)
    }
    with open(conf_matrix_file, 'w') as f:
        json.dump(pred_json, f, indent=2)

Streaming output truncated to the last 5000 lines.
seq_2190521  Cacao            82.03%      
seq_2190522  Cacao            77.25%      
seq_2190523  Cacao            73.90%      
seq_2190524  Cacao            75.90%      
seq_2190525  Cacao            66.57%      
seq_2190526  Cacao            80.95%      
seq_2190527  Cacao            80.53%      
seq_2190528  Cacao            78.25%      
seq_2190529  Cacao            70.26%      
seq_2190530  Cacao            66.02%      
seq_2190531  Cacao            76.10%      
seq_2190532  Cacao            85.94%      
seq_2190533  Cacao            79.48%      
seq_2190534  Cacao            77.92%      
seq_2190535  Cacao            74.10%      
seq_2190536  Cacao            82.66%      
seq_2190537  Cacao            85.57%      
seq_2190538  Cacao            84.58%      
seq_2190539  Cacao            69.00%      
seq_2190540  Cacao            84.03%      
seq_2190541  Cacao            85.60%      
seq_2190542  Cacao            83.94%      
seq

In [9]:
from google.colab import files

# Cambia este nombre al archivo que deseas descargar
archivo_json = '/content/confidencias.json'

# Descarga el archivo a tu computadora
files.download(archivo_json)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
import psutil
import shutil
import time
import datetime
import os
from google.colab import files

# Tiempo de uso del runtime
uptime_seconds = time.time() - psutil.boot_time()
uptime_string = str(datetime.timedelta(seconds=int(uptime_seconds)))

# Disco
total_disk, used_disk, free_disk = shutil.disk_usage('/')
disk_stats = f"{used_disk // (1024**3)} GB usados de {total_disk // (1024**3)} GB"

# RAM del sistema
ram = psutil.virtual_memory()
ram_stats = f"{ram.used // (1024**2)} MB usados de {ram.total // (1024**2)} MB"

# GPU RAM (requiere NVIDIA GPU)
gpu_stats = ""
try:
    gpu_info = os.popen("nvidia-smi --query-gpu=memory.used,memory.total --format=csv,nounits,noheader").read().strip()
    if gpu_info:
        used_mem, total_mem = map(int, gpu_info.split(','))
        gpu_stats = f"{used_mem} MB usados de {total_mem} MB"
    else:
        gpu_stats = "GPU no disponible"
except:
    gpu_stats = "Error consultando GPU"

# Guardar en archivo .txt
contenido = f"""📊 Estadísticas del entorno de Google Colab

⏱️ Tiempo de uso del runtime: {uptime_string}
💾 Disco: {disk_stats}
🧠 RAM del sistema: {ram_stats}
🎮 RAM de la GPU: {gpu_stats}
"""

with open("colab_stats.txt", "w") as f:
    f.write(contenido)

# Descargar archivo
files.download("colab_stats.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
import json
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import pandas as pd

# Ruta al archivo confidencias.json
input_file = "/content/confidencias.json"

# Cargar datos del archivo
with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

# Simulación de etiquetas verdaderas (ajusta con tus datos reales si los tienes)
y_true = ["Cacao" if i % 2 == 0 else "NotCacao" for i in range(len(data))]

# Obtener predicciones desde el campo "Predicted"
y_pred = [entry["Predicted"].replace("No Cacao", "NotCacao") for entry in data.values()]

# Cálculo de métricas
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, pos_label="Cacao")
recall = recall_score(y_true, y_pred, pos_label="Cacao")
f1 = f1_score(y_true, y_pred, pos_label="Cacao")
cm = confusion_matrix(y_true, y_pred, labels=["Cacao", "NotCacao"])
report = classification_report(y_true, y_pred, target_names=["Cacao", "NotCacao"], output_dict=True)

# Mostrar resultados
print("\n📊 MÉTRICAS DE CLASIFICACIÓN (Cacao vs NotCacao)")
print(f"✔ Accuracy:          {accuracy:.2%}")
print(f"✔ Precision (Cacao): {precision:.2%}")
print(f"✔ Recall (Cacao):    {recall:.2%}")
print(f"✔ F1-score (Cacao):  {f1:.2%}")

print("\n🧮 MATRIZ DE CONFUSIÓN")
print(pd.DataFrame(cm, index=["Real: Cacao", "Real: NotCacao"], columns=["Predicho: Cacao", "Predicho: NotCacao"]))

print("\n📋 REPORTE DETALLADO")
df_report = pd.DataFrame(report).transpose()
print(df_report)



📊 MÉTRICAS DE CLASIFICACIÓN (Cacao vs NotCacao)
✔ Accuracy:          50.02%
✔ Precision (Cacao): 50.01%
✔ Recall (Cacao):    98.51%
✔ F1-score (Cacao):  66.34%

🧮 MATRIZ DE CONFUSIÓN
                Predicho: Cacao  Predicho: NotCacao
Real: Cacao             1081404               16357
Real: NotCacao          1080881               16879

📋 REPORTE DETALLADO
              precision    recall  f1-score       support
Cacao          0.500121  0.985100  0.663429  1.097761e+06
NotCacao       0.507853  0.015376  0.029848  1.097760e+06
accuracy       0.500238  0.500238  0.500238  5.002380e-01
macro avg      0.503987  0.500238  0.346638  2.195521e+06
weighted avg   0.503987  0.500238  0.346638  2.195521e+06
